In [0]:
#  **Objectif** : Créer des vues SQL optimisées pour dashboards Power BI
#  - Vue agrégée KPIs principaux
#  - Vue détaillée patients à risque
#  - Vue tendances temporelles
#  - Vue comparaison modèles
#  - Vue analyse par catégorie
#  - Pipeline automatisé de rafraîchissement
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("IMPORTS REUSSIS")
print("="*70)

PREDICTIONS_TABLE = "workspace.default.hospital_readmissions_predictions"
MODELING_TABLE = "workspace.default.hospital_readmissions_modeling"
DATAMART_SCHEMA = "workspace.default"

print("Configuration :")
print(f"Source prédictions : {PREDICTIONS_TABLE}")
print(f"Source modeling    : {MODELING_TABLE}")
print(f"Schema datamart    : {DATAMART_SCHEMA}")


IMPORTS REUSSIS
Configuration :
Source prédictions : workspace.default.hospital_readmissions_predictions
Source modeling    : workspace.default.hospital_readmissions_modeling
Schema datamart    : workspace.default


In [0]:
# CELLULE 2 : VÉRIFICATION DES DONNÉES SOURCES
print("="*70)
print("VERIFICATION DES DONNEES SOURCES")
print("="*70)

df_predictions = spark.table(PREDICTIONS_TABLE)
count_predictions = df_predictions.count()

print(f"Table prédictions : {count_predictions} lignes")

print("Colonnes disponibles :")
for field in df_predictions.schema.fields:
    print(f"{field.name} : {field.dataType}")

stats = df_predictions.select(
    F.min("prediction_timestamp").alias("premiere_prediction"),
    F.max("prediction_timestamp").alias("derniere_prediction"),
    F.countDistinct("patient_id").alias("patients_uniques")
).collect()[0]

print("Statistiques :")
print(f"Premiere prediction : {stats['premiere_prediction']}")
print(f"Derniere prediction : {stats['derniere_prediction']}")
print(f"Patients uniques    : {stats['patients_uniques']}")


VERIFICATION DES DONNEES SOURCES
Table prédictions : 550 lignes
Colonnes disponibles :
prediction : LongType()
risk_probability : DoubleType()
risk_score_100 : DoubleType()
risk_category : StringType()
patient_id : StringType()
readmission_reelle : BooleanType()
prediction_timestamp : TimestampType()
model_name : StringType()
model_version : StringType()
Statistiques :
Premiere prediction : 2025-12-25 11:10:34.363646
Derniere prediction : 2026-01-24 22:53:39.565736
Patients uniques    : 100


In [0]:
df_predictions.printSchema

<bound method DataFrame.printSchema of DataFrame[prediction: bigint, risk_probability: double, risk_score_100: double, risk_category: string, patient_id: string, readmission_reelle: boolean, prediction_timestamp: timestamp, model_name: string, model_version: string]>

In [0]:

# CELLULE 3 : VUE 1 - KPIs PRINCIPAUX (Agrégation Quotidienne)
view_name_kpis = f"{DATAMART_SCHEMA}.vw_readmissions_kpis_daily"

spark.sql(f"""
CREATE OR REPLACE VIEW {view_name_kpis} AS
SELECT
    DATE(prediction_timestamp) AS prediction_date,
    YEAR(prediction_timestamp) AS annee,
    MONTH(prediction_timestamp) AS mois,
    DAYOFWEEK(prediction_timestamp) AS jour_semaine,
    model_name,
    model_version,
    COUNT(*) AS total_patients,
    COUNT(DISTINCT patient_id) AS patients_uniques,
    ROUND(AVG(risk_score_100), 2) AS score_risque_moyen,
    ROUND(MIN(risk_score_100), 2) AS score_risque_min,
    ROUND(MAX(risk_score_100), 2) AS score_risque_max,
    ROUND(STDDEV(risk_score_100), 2) AS score_risque_ecart_type,
    SUM(CASE WHEN risk_category = 'Élevé' THEN 1 ELSE 0 END) AS nb_risque_eleve,
    SUM(CASE WHEN risk_category = 'Moyen' THEN 1 ELSE 0 END) AS nb_risque_moyen,
    SUM(CASE WHEN risk_category = 'Faible' THEN 1 ELSE 0 END) AS nb_risque_faible,
    ROUND(100.0 * SUM(CASE WHEN risk_category = 'Élevé' THEN 1 ELSE 0 END)/COUNT(*),2) AS pct_risque_eleve,
    ROUND(100.0 * SUM(CASE WHEN prediction = 1 THEN 1 ELSE 0 END)/COUNT(*),2) AS taux_prediction_readmission,
    ROUND(100.0 * SUM(CASE WHEN readmission_reelle = true THEN 1 ELSE 0 END)/COUNT(*),2) AS taux_readmission_reel,
    ROUND(100.0 * SUM(CASE WHEN prediction = CAST(readmission_reelle AS INT) THEN 1 ELSE 0 END)/COUNT(*),2) AS accuracy_pct,
    CURRENT_TIMESTAMP() AS vue_created_at
FROM {PREDICTIONS_TABLE}
GROUP BY
    DATE(prediction_timestamp),
    YEAR(prediction_timestamp),
    MONTH(prediction_timestamp),
    DAYOFWEEK(prediction_timestamp),
    model_name,
    model_version
""")


DataFrame[]

In [0]:
# CELLULE 4 BIS : VUE 2 - PATIENTS À HAUT RISQUE 

view_name_high_risk = f"{DATAMART_SCHEMA}.vw_readmissions_high_risk_patients"

print("=" * 70)
print("CRÉATION VUE : PATIENTS À HAUT RISQUE (CORRIGÉE)")
print("=" * 70)

spark.sql(f"""
CREATE OR REPLACE VIEW {view_name_high_risk} AS
SELECT
    patient_id,
    risk_score_100,
    risk_probability,
    risk_category,
    prediction AS prediction_binaire,
    readmission_reelle,
    prediction_timestamp,
    DATE(prediction_timestamp) AS prediction_date,
    model_name,
    model_version,
    
    -- Niveau d'alerte basé sur le SCORE (pas sur la catégorie)
    CASE
        WHEN risk_score_100 >= 90 THEN 'URGENT'
        WHEN risk_score_100 >= 80 THEN 'PRIORITAIRE'
        WHEN risk_score_100 >= 70 THEN 'À SURVEILLER'
        ELSE 'NORMAL'
    END AS niveau_alerte,
    
    -- Rang quotidien par score
    ROW_NUMBER() OVER (
        PARTITION BY DATE(prediction_timestamp)
        ORDER BY risk_score_100 DESC
    ) AS rang_quotidien,
    
    CURRENT_TIMESTAMP() AS vue_created_at
FROM {PREDICTIONS_TABLE}
-- CHANGEMENT CLÉ : Ne plus filtrer uniquement sur 'Élevé'
-- On inclut tous les patients avec score >= 70 (À SURVEILLER minimum)
WHERE risk_score_100 >= 70  -- Au lieu de : WHERE risk_category = 'Élevé'
ORDER BY risk_score_100 DESC
""")

print(" Vue vw_readmissions_high_risk_patients créée avec succès")

# Vérifier le contenu
print("\n Distribution des niveaux d'alerte dans la vue :")
df_verification = spark.sql(f"""
SELECT 
    niveau_alerte,
    COUNT(*) as nombre_patients,
    MIN(risk_score_100) as score_min,
    MAX(risk_score_100) as score_max,
    AVG(risk_score_100) as score_moyen
FROM {view_name_high_risk}
GROUP BY niveau_alerte
ORDER BY score_moyen DESC
""")
display(df_verification)

# Vérifier aussi par catégorie de risque originale
print("\n Répartition par catégorie de risque originale :")
df_par_categorie = spark.sql(f"""
SELECT 
    risk_category,
    niveau_alerte,
    COUNT(*) as nombre_patients
FROM {view_name_high_risk}
GROUP BY risk_category, niveau_alerte
ORDER BY niveau_alerte, risk_category
""")
display(df_par_categorie)

# Afficher quelques exemples
print("\n Exemples de patients par niveau d'alerte :")
df_exemples = spark.sql(f"""
SELECT 
    niveau_alerte,
    patient_id,
    risk_score_100,
    risk_category,
    prediction_date
FROM {view_name_high_risk}
WHERE rang_quotidien <= 3
ORDER BY niveau_alerte, risk_score_100 DESC
LIMIT 10
""")
display(df_exemples)

print("\n" + "=" * 70)
print("VUE MISE À JOUR")
print("=" * 70)

CRÉATION VUE : PATIENTS À HAUT RISQUE (CORRIGÉE)
 Vue vw_readmissions_high_risk_patients créée avec succès

 Distribution des niveaux d'alerte dans la vue :


niveau_alerte,nombre_patients,score_min,score_max,score_moyen
À SURVEILLER,12,70.08031852246943,71.7091190048616,71.55677141337917



 Répartition par catégorie de risque originale :


risk_category,niveau_alerte,nombre_patients
Élevé,À SURVEILLER,12



 Exemples de patients par niveau d'alerte :


niveau_alerte,patient_id,risk_score_100,risk_category,prediction_date
À SURVEILLER,PAT_040627,71.7091190048616,Élevé,2026-01-24
À SURVEILLER,PAT_040627,71.7091190048616,Élevé,2026-01-24
À SURVEILLER,PAT_040627,71.7091190048616,Élevé,2026-01-21
À SURVEILLER,PAT_040627,71.7091190048616,Élevé,2026-01-21
À SURVEILLER,PAT_040627,71.7091190048616,Élevé,2026-01-21
À SURVEILLER,PAT_040627,71.7091190048616,Élevé,2026-01-23
À SURVEILLER,PAT_040627,71.7091190048616,Élevé,2025-12-25
À SURVEILLER,PAT_040627,71.7091190048616,Élevé,2026-01-24



VUE MISE À JOUR


In [0]:
# CELLULE 5 : VUE 3 - TENDANCES TEMPORELLES
view_name_trends = f"{DATAMART_SCHEMA}.vw_readmissions_trends"

spark.sql(f"""
CREATE OR REPLACE VIEW {view_name_trends} AS
WITH daily_stats AS (
    SELECT
        DATE(prediction_timestamp) AS prediction_date,
        COUNT(*) AS total_patients,
        AVG(risk_score_100) AS score_moyen,
        SUM(CASE WHEN risk_category = 'Élevé' THEN 1 ELSE 0 END) AS nb_risque_eleve,
        SUM(CASE WHEN prediction = 1 THEN 1 ELSE 0 END) AS nb_predits_readmis
    FROM {PREDICTIONS_TABLE}
    GROUP BY DATE(prediction_timestamp)
)
SELECT
    prediction_date,
    total_patients,
    ROUND(score_moyen,2) AS score_risque_moyen,
    nb_risque_eleve,
    nb_predits_readmis,
    ROUND(AVG(score_moyen) OVER (ORDER BY prediction_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),2) AS score_moyen_7j,
    SUM(total_patients) OVER (ORDER BY prediction_date) AS patients_cumules,
    CURRENT_TIMESTAMP() AS vue_created_at
FROM daily_stats
""")

DataFrame[]

In [0]:
#  CELLULE 6 : VUE 4 - COMPARAISON VERSIONS MODÈLES
view_name_model_comparison = f"{DATAMART_SCHEMA}.vw_readmissions_model_comparison"

spark.sql(f"""
CREATE OR REPLACE VIEW {view_name_model_comparison} AS
SELECT
    model_name,
    model_version,
    MIN(DATE(prediction_timestamp)) AS premiere_utilisation,
    MAX(DATE(prediction_timestamp)) AS derniere_utilisation,
    COUNT(*) AS total_predictions,
    COUNT(DISTINCT patient_id) AS patients_uniques,
    ROUND(AVG(risk_score_100),2) AS score_moyen,
    ROUND(100.0 * SUM(CASE WHEN prediction = CAST(readmission_reelle AS INT) THEN 1 ELSE 0 END)/COUNT(*),2) AS accuracy,
    CURRENT_TIMESTAMP() AS vue_created_at
FROM {PREDICTIONS_TABLE}
GROUP BY model_name, model_version
""")


DataFrame[]

In [0]:
view_name_by_category = f"{DATAMART_SCHEMA}.vw_readmissions_by_risk_category"

spark.sql(f"""
CREATE OR REPLACE VIEW {view_name_by_category} AS
SELECT
    risk_category,
    COUNT(*) AS total_patients,
    ROUND(AVG(risk_score_100),2) AS score_moyen,
    ROUND(PERCENTILE(risk_score_100,0.5),2) AS score_mediane,
    ROUND(100.0 * SUM(CASE WHEN readmission_reelle = true THEN 1 ELSE 0 END)/COUNT(*),2) AS taux_readmission_reel,
    ROUND(100.0 * SUM(CASE WHEN prediction = CAST(readmission_reelle AS INT) THEN 1 ELSE 0 END)/COUNT(*),2) AS accuracy,
    
    -- Ajout des métriques de matrice de confusion
    SUM(CASE 
        WHEN risk_category = 'Élevé' AND readmission_reelle = true THEN 1 
        ELSE 0 
    END) AS vrais_positifs,
    
    SUM(CASE 
        WHEN risk_category = 'Faible' AND readmission_reelle = false THEN 1 
        ELSE 0 
    END) AS vrais_negatifs,
    
    SUM(CASE 
        WHEN risk_category = 'Élevé' AND readmission_reelle = false THEN 1 
        ELSE 0 
    END) AS faux_positifs,
    
    SUM(CASE 
        WHEN risk_category = 'Faible' AND readmission_reelle = true THEN 1 
        ELSE 0 
    END) AS faux_negatifs,
    
    CURRENT_TIMESTAMP() AS vue_created_at
FROM {PREDICTIONS_TABLE}
GROUP BY risk_category
""")

DataFrame[]

In [0]:
# CELLULE 8 : FONCTION DE RAFRAÎCHISSEMENT AUTOMATIQUE
def refresh_powerbi_datamart():
    views = [
        "vw_readmissions_kpis_daily",
        "vw_readmissions_high_risk_patients",
        "vw_readmissions_trends",
        "vw_readmissions_model_comparison",
        "vw_readmissions_by_risk_category",
        "vw_readmissions_confusion_matrix"
    ]
    for v in views:
        count = spark.sql(f"SELECT COUNT(*) FROM {DATAMART_SCHEMA}.{v}").collect()[0][0]
        print(f"{v} : {count} lignes")


In [0]:
# CELLULE 9 : VUE CATALOGUE (Documentation)
spark.sql(f"""
CREATE OR REPLACE VIEW {DATAMART_SCHEMA}.vw_datamart_catalog AS
SELECT 'vw_readmissions_kpis_daily' AS vue_name,
       'KPIs quotidiens agrégés' AS description,
       CURRENT_TIMESTAMP() AS created_at
UNION ALL
SELECT 'vw_readmissions_high_risk_patients',
       'Patients à haut risque',
       CURRENT_TIMESTAMP()
UNION ALL
SELECT 'vw_readmissions_trends',
       'Tendances temporelles',
       CURRENT_TIMESTAMP()
UNION ALL
SELECT 'vw_readmissions_model_comparison',
       'Comparaison des modèles',
       CURRENT_TIMESTAMP()
UNION ALL
SELECT 'vw_readmissions_by_risk_category',
       'Analyse par catégorie',
       CURRENT_TIMESTAMP()
UNION ALL
SELECT 'vw_readmissions_confusion_matrix',
       'Matrice de confusion',
       CURRENT_TIMESTAMP()


""")


DataFrame[]

In [0]:
# CELLULE 10 : CONNEXION POWER BI (Instructions)
print("="*70)
print("INSTRUCTIONS CONNEXION POWER BI")
print("="*70)

INSTRUCTIONS CONNEXION POWER BI


In [0]:
# CELLULE 11 : STATISTIQUES DU DATAMART
print("="*70)
print("STATISTIQUES DU DATAMART")
print("="*70)
def get_view_stats(view_name):
    try:
        count = spark.sql(f"SELECT COUNT(*) as cnt FROM {view_name}").collect()[0]['cnt']
        df = spark.sql(f"SELECT * FROM {view_name} LIMIT 1")
        num_cols = len(df.columns)
        return {
            "vue": view_name.split(".")[-1],
            "lignes": count,
            "colonnes": num_cols,
            "status": "OK"
        }
    except Exception as e:
        return {
            "vue": view_name.split(".")[-1],
            "lignes": 0,
            "colonnes": 0,
            "status": str(e)[:50]
        }
views_list = [
    f"{DATAMART_SCHEMA}.vw_readmissions_kpis_daily",
    f"{DATAMART_SCHEMA}.vw_readmissions_high_risk_patients",
    f"{DATAMART_SCHEMA}.vw_readmissions_trends",
    f"{DATAMART_SCHEMA}.vw_readmissions_model_comparison",
    f"{DATAMART_SCHEMA}.vw_readmissions_by_risk_category",
    f"{DATAMART_SCHEMA}.vw_readmissions_confusion_matrix"
]
stats_list = [get_view_stats(v) for v in views_list]
df_stats = pd.DataFrame(stats_list)
print("\nRESUME DES VUES")
print("-"*70)
print(f"{'Vue':<45} {'Lignes':>10} {'Colonnes':>10}")
print("-"*70)
for _, row in df_stats.iterrows():
    print(f"{row['vue']:<45} {row['lignes']:>10,} {row['colonnes']:>10}")
print("-"*70)
print(f"{'TOTAL':<45} {df_stats['lignes'].sum():>10,}")
total_predictions = spark.table(PREDICTIONS_TABLE).count()
print("\nSTATISTIQUES GLOBALES")
print(f"Source predictions : {total_predictions:,}")
print(f"Nombre de vues     : {len(views_list)}")
print(f"Date creation     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

STATISTIQUES DU DATAMART

RESUME DES VUES
----------------------------------------------------------------------
Vue                                               Lignes   Colonnes
----------------------------------------------------------------------
vw_readmissions_kpis_daily                             4         20
vw_readmissions_high_risk_patients                    12         13
vw_readmissions_trends                                 4          8
vw_readmissions_model_comparison                       1          9
vw_readmissions_by_risk_category                       3          7
vw_readmissions_confusion_matrix                       1          6
----------------------------------------------------------------------
TOTAL                                                 25

STATISTIQUES GLOBALES
Source predictions : 550
Nombre de vues     : 6
Date creation     : 2026-01-24 23:03:04


In [0]:
# CELLULE 12 : PIPELINE DE BATCH SCORING (CORRIGÉ)
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType, StringType, DoubleType, BooleanType, IntegerType, LongType
import mlflow
from mlflow.tracking import MlflowClient
def batch_score_new_patients(n_patients=50, model_alias="champion"):
    print("="*70)
    print("BATCH SCORING AUTOMATISE")
    print("="*70)
    # ============================================================
    # 1. CHARGEMENT DU MODÈLE DEPUIS MLFLOW
    # ============================================================
    model_name = "hospital_readmissions_model_mlflow"
    try:
        model_uri = f"models:/{model_name}@{model_alias}"
        model = mlflow.sklearn.load_model(model_uri)
        print(f" Modèle chargé : {model_name}@{model_alias}")
    except:
        model_name = f"workspace.default.{model_name}"
        model_uri = f"models:/{model_name}@{model_alias}"
        model = mlflow.sklearn.load_model(model_uri)
        print(f" Modèle chargé : {model_name}@{model_alias}")
    # Récupérer la version du modèle
    client = MlflowClient()
    try:
        model_details = client.get_registered_model(model_name)
        model_version = model_details.aliases.get(model_alias, "Unknown")
    except:
        model_version = "Unknown"
    print(f"  Version : {model_version}")
    # ============================================================
    # 2. CHARGEMENT DES DONNÉES (DÉJÀ TRANSFORMÉES)
    # ============================================================
    df_pd = spark.table(MODELING_TABLE).toPandas()
    print(f"\n Données chargées depuis {MODELING_TABLE}")
    print(f"  Nombre de lignes : {len(df_pd):,}")
    # ============================================================
    # 3. PRÉPARATION DES FEATURES
    # ============================================================
    patient_ids = df_pd["patient_id"]
    X = df_pd.drop(columns=["patient_id", "readmission_30j"])
    y = df_pd["readmission_30j"].astype(bool)
    print(f"\n Features : {X.shape[1]} colonnes")
    # Sélection aléatoire
    np.random.seed(42)
    idx = np.random.choice(len(X), min(n_patients, len(X)), replace=False)

    X_to_score = X.iloc[idx].reset_index(drop=True)
    patient_ids_selected = patient_ids.iloc[idx].reset_index(drop=True)
    y_true = y.iloc[idx].reset_index(drop=True)
    print(f" {len(X_to_score)} patients sélectionnés")
    # ============================================================
    # 4. SCORING
    # ============================================================
    print("\n Calcul des prédictions...")
    
    probas = model.predict_proba(X_to_score)
    predictions_binary = model.predict(X_to_score)
    risk_proba = probas[:, 1]
    risk_score_100 = risk_proba * 100
    def categorize_risk(score):
        if score < 40:
            return "Faible"
        elif score < 70:
            return "Moyen"
        else:
            return "Élevé"
    predictions = pd.DataFrame({
        "patient_id": patient_ids_selected.astype(str),  
        "prediction": predictions_binary.astype(np.int64), 
        "risk_probability": risk_proba.astype(np.float64),  
        "risk_score_100": risk_score_100.astype(np.float64),  
        "risk_category": [categorize_risk(s) for s in risk_score_100],  
        "readmission_reelle": y_true.astype(bool), 
        "prediction_timestamp": pd.Timestamp.now(),  
        "model_name": str(model_name), 
        "model_version": str(model_version)  
    })
    print(f" Prédictions calculées")
    # 5. VÉRIFIER LE SCHÉMA DE LA TABLE EXISTANTE
    print("\n Vérification du schéma...")
    existing_schema = spark.table(PREDICTIONS_TABLE).schema
    # Afficher les types attendus
    print("Types attendus dans la table :")
    for field in existing_schema.fields:
        print(f"  • {field.name}: {field.dataType}")
    # 6. SAUVEGARDE AVEC TYPES EXPLICITES
    spark_df = spark.createDataFrame(predictions)
    print("\nTypes avant conversion :")
    spark_df.printSchema()
    spark_df = (
        spark_df
        .withColumn("patient_id", col("patient_id").cast(StringType()))
        .withColumn("prediction", col("prediction").cast(LongType())) 
        .withColumn("risk_probability", col("risk_probability").cast(DoubleType()))
        .withColumn("risk_score_100", col("risk_score_100").cast(DoubleType()))
        .withColumn("risk_category", col("risk_category").cast(StringType()))
        .withColumn("readmission_reelle", col("readmission_reelle").cast(BooleanType()))
        .withColumn("prediction_timestamp", col("prediction_timestamp").cast(TimestampType()))
        .withColumn("model_name", col("model_name").cast(StringType()))
        .withColumn("model_version", col("model_version").cast(StringType()))
    )
    print("\nTypes après conversion :")
    spark_df.printSchema()
    # Sauvegarder
    try:
        spark_df.write.format("delta").mode("append").saveAsTable(PREDICTIONS_TABLE)
        print(f"\nPrédictions sauvegardées dans {PREDICTIONS_TABLE}")
        print(f"  Lignes ajoutées : {len(predictions)}")
    except Exception as e:
        print(f"\n Erreur lors de la sauvegarde : {e}")
        raise
    # ============================================================
    # 7. STATISTIQUES
    # ============================================================
    print("\n Distribution des risques :")
    for cat in ["Faible", "Moyen", "Élevé"]:
        count = (predictions["risk_category"] == cat).sum()
        pct = count / len(predictions) * 100
        print(f"  {cat:<10} : {count:>3} ({pct:>5.1f}%)")
    high_risk = predictions[predictions["risk_category"] == "Élevé"]
    if len(high_risk) > 0:
        print(f"\n {len(high_risk)} patient(s) à HAUT RISQUE")
    # ============================================================
    # 8. RAFRAÎCHIR POWER BI
    # ============================================================
    try:
        refresh_powerbi_datamart()
        print(f"\n Power BI rafraîchi")
    except Exception as e:
        print(f"\n Power BI non rafraîchi: {e}")
    print("\n" + "="*70)
    print("BATCH SCORING TERMINÉ")
    print("="*70)
    return predictions
print(" Fonction batch_score_new_patients() corrigée")

 Fonction batch_score_new_patients() corrigée


In [0]:
# CELLULE 13 : TEST DU PIPELINE COMPLET
print("=" * 70)
print("TEST DU PIPELINE COMPLET")
print("=" * 70)

try:
    # Exécution du batch scoring
    predictions_test = batch_score_new_patients(n_patients=50)
    
    # Vérification finale
    print("\n" + "=" * 70)
    print("VÉRIFICATION FINALE")
    print("=" * 70)
    
    total_in_table = spark.table(PREDICTIONS_TABLE).count()
    print(f" Total de prédictions en base : {total_in_table:,}")
    
    # Statistiques supplémentaires
    if predictions_test is not None:
        print(f" Nouvelles prédictions : {len(predictions_test)}")
        print(f" Taux de réadmission prédit : {predictions_test['prediction'].mean():.1%}")
        print(f" Score de risque moyen : {predictions_test['risk_score_100'].mean():.1f}/100")
    # Affichage des premières prédictions
    print("\n" + "=" * 50)
    print("APERÇU DES PRÉDICTIONS")
    print("=" * 50)
    display(predictions_test.head(5))
    # Vérification des patients à haut risque
    high_risk = predictions_test[predictions_test['risk_category'] == 'Élevé']
    if len(high_risk) > 0:
        print(f"\n  {len(high_risk)} patient(s) à HAUT RISQUE détecté(s)")
        print("\nDétails des patients à haut risque :")
        display(high_risk[['patient_id', 'risk_score_100', 'risk_probability']])
    print("\n" + "=" * 70)
    print("✓ TEST TERMINÉ AVEC SUCCÈS")
    print("=" * 70)
except Exception as e:
    print("\n" + "=" * 70)
    print(" ERREUR LORS DU TEST")
    print("=" * 70)
    print(f"Erreur : {str(e)}")
    import traceback
    traceback.print_exc()

TEST DU PIPELINE COMPLET
BATCH SCORING AUTOMATISE
 Modèle chargé : hospital_readmissions_model_mlflow@champion
  Version : 5

 Données chargées depuis workspace.default.hospital_readmissions_modeling
  Nombre de lignes : 69,750

 Features : 7 colonnes
 50 patients sélectionnés

 Calcul des prédictions...
 Prédictions calculées

 Vérification du schéma...
Types attendus dans la table :
  • prediction: LongType()
  • risk_probability: DoubleType()
  • risk_score_100: DoubleType()
  • risk_category: StringType()
  • patient_id: StringType()
  • readmission_reelle: BooleanType()
  • prediction_timestamp: TimestampType()
  • model_name: StringType()
  • model_version: StringType()

Types avant conversion :
root
 |-- patient_id: string (nullable = true)
 |-- prediction: long (nullable = true)
 |-- risk_probability: double (nullable = true)
 |-- risk_score_100: double (nullable = true)
 |-- risk_category: string (nullable = true)
 |-- readmission_reelle: boolean (nullable = true)
 |-- predict

patient_id,prediction,risk_probability,risk_score_100,risk_category,readmission_reelle,prediction_timestamp,model_name,model_version
PAT_004411,1,0.5509565249724736,55.09565249724736,Moyen,false,2026-01-24T23:03:09.835Z,hospital_readmissions_model_mlflow,5
PAT_061311,0,0.4079698525633367,40.79698525633367,Moyen,true,2026-01-24T23:03:09.835Z,hospital_readmissions_model_mlflow,5
PAT_069370,1,0.668275935719723,66.8275935719723,Moyen,false,2026-01-24T23:03:09.835Z,hospital_readmissions_model_mlflow,5
PAT_001180,0,0.44234685639320664,44.234685639320666,Moyen,false,2026-01-24T23:03:09.835Z,hospital_readmissions_model_mlflow,5
PAT_020969,0,0.3506019636982881,35.06019636982881,Faible,true,2026-01-24T23:03:09.835Z,hospital_readmissions_model_mlflow,5



  1 patient(s) à HAUT RISQUE détecté(s)

Détails des patients à haut risque :


patient_id,risk_score_100,risk_probability
PAT_040627,71.7091190048616,0.717091190048616



✓ TEST TERMINÉ AVEC SUCCÈS


In [0]:
# CELLULE 14 : VALIDATION DES DONNÉES
print("=" * 70)
print("VALIDATION DES DONNÉES")
print("=" * 70)
# Charger toutes les prédictions
all_predictions = spark.table(PREDICTIONS_TABLE).toPandas()

print(f"\n Statistiques globales ({len(all_predictions)} prédictions)")
print("-" * 70)

# Distribution des catégories de risque
risk_dist = all_predictions['risk_category'].value_counts()
print("\nDistribution des risques :")
for category, count in risk_dist.items():
    pct = (count / len(all_predictions)) * 100
    print(f"  {category:12s} : {count:5d} ({pct:5.1f}%)")
# Statistiques des scores
print(f"\nScore de risque (0-100) :")
print(f"  Minimum  : {all_predictions['risk_score_100'].min():.1f}")
print(f"  Maximum  : {all_predictions['risk_score_100'].max():.1f}")
print(f"  Moyenne  : {all_predictions['risk_score_100'].mean():.1f}")
print(f"  Médiane  : {all_predictions['risk_score_100'].median():.1f}")
# Taux de réadmission
if 'readmission_reelle' in all_predictions.columns:
    actual_rate = all_predictions['readmission_reelle'].mean()
    predicted_rate = all_predictions['prediction'].mean()
    print(f"\nTaux de réadmission :")
    print(f"  Réel     : {actual_rate:.1%}")
    print(f"  Prédit   : {predicted_rate:.1%}")

# Versions du modèle utilisées
model_versions = all_predictions['model_version'].value_counts()
print(f"\nVersions du modèle utilisées :")
for version, count in model_versions.items():
    print(f"  Version {version} : {count} prédictions")

print("=" * 70)

VALIDATION DES DONNÉES

 Statistiques globales (600 prédictions)
----------------------------------------------------------------------

Distribution des risques :
  Moyen        :   391 ( 65.2%)
  Faible       :   196 ( 32.7%)
  Élevé        :    13 (  2.2%)

Score de risque (0-100) :
  Minimum  : 29.6
  Maximum  : 71.7
  Moyenne  : 47.4
  Médiane  : 44.2

Taux de réadmission :
  Réel     : 23.5%
  Prédit   : 40.0%

Versions du modèle utilisées :
  Version 5 : 600 prédictions


In [0]:
# CELLULE 15 : CONFIGURATION DES ALERTES E-MAIL
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime
import pandas as pd

# Configuration e-mail (à adapter selon votre serveur)
EMAIL_CONFIG = {
    'smtp_server': 'smtp.gmail.com',  # ou smtp.office365.com pour Outlook
    'smtp_port': 587,
    'sender_email': 'gueyeamdy457@gmail.com',
    'sender_password': 'gemp mwtn vmle svlt',  #  À stocker dans Databricks Secrets
    'recipient_emails': [
        'contactgueye@gmail.com',
        'amgtechsavvy@gmail.com'
    ]
}

def send_high_risk_alert(high_risk_patients, config=EMAIL_CONFIG):
    """
    Envoie un e-mail d'alerte pour les patients à haut risque
    
    Args:
        high_risk_patients (pd.DataFrame): DataFrame des patients à haut risque
        config (dict): Configuration e-mail
    """
    
    if len(high_risk_patients) == 0:
        print("✓ Aucun patient à haut risque - Pas d'alerte envoyée")
        return
    
    try:
        # Création du message
        msg = MIMEMultipart('alternative')
        msg['From'] = config['sender_email']
        msg['To'] = ', '.join(config['recipient_emails'])
        msg['Subject'] = f"🚨 ALERTE : {len(high_risk_patients)} patient(s) à HAUT RISQUE de réadmission"
        
        # Corps de l'e-mail en HTML
        html_body = create_email_html(high_risk_patients)
        
        # Attacher le corps HTML
        msg.attach(MIMEText(html_body, 'html'))
        
        # Connexion et envoi
        with smtplib.SMTP(config['smtp_server'], config['smtp_port']) as server:
            server.starttls()
            server.login(config['sender_email'], config['sender_password'])
            server.send_message(msg)
        
        print(f" E-mail d'alerte envoyé avec succès à {len(config['recipient_emails'])} destinataire(s)")
        print(f"  Patients concernés : {len(high_risk_patients)}")
        
    except Exception as e:
        print(f" Erreur lors de l'envoi de l'e-mail : {str(e)}")
        raise

def create_email_html(high_risk_patients):
    """
    Crée le contenu HTML de l'e-mail d'alerte
    """
    
    # Tri par score de risque décroissant
    patients_sorted = high_risk_patients.sort_values('risk_score_100', ascending=False)
    
    # En-tête
    html = f"""
    <html>
    <head>
        <style>
            body {{ font-family: Arial, sans-serif; }}
            .header {{ background-color: #d32f2f; color: white; padding: 20px; text-align: center; }}
            .content {{ padding: 20px; }}
            .stats {{ background-color: #fff3cd; padding: 15px; margin: 20px 0; border-radius: 5px; }}
            table {{ border-collapse: collapse; width: 100%; margin-top: 20px; }}
            th {{ background-color: #424242; color: white; padding: 12px; text-align: left; }}
            td {{ padding: 10px; border-bottom: 1px solid #ddd; }}
            tr:hover {{ background-color: #f5f5f5; }}
            .risk-high {{ color: #d32f2f; font-weight: bold; }}
            .footer {{ margin-top: 30px; padding: 15px; background-color: #f5f5f5; font-size: 12px; }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>🚨 ALERTE PATIENTS À HAUT RISQUE</h1>
            <p>Système de Prédiction de Réadmission Hospitalière</p>
        </div>
        
        <div class="content">
            <div class="stats">
                <h2>Résumé</h2>
                <ul>
                    <li><strong>{len(patients_sorted)}</strong> patient(s) identifié(s) à <span class="risk-high">HAUT RISQUE</span></li>
                    <li><strong>Score de risque moyen :</strong> {patients_sorted['risk_score_100'].mean():.1f}/100</li>
                    <li><strong>Score maximal :</strong> {patients_sorted['risk_score_100'].max():.1f}/100</li>
                    <li><strong>Date d'analyse :</strong> {datetime.now().strftime('%d/%m/%Y à %H:%M')}</li>
                </ul>
            </div>
            
            <h2>👥 Liste des patients prioritaires</h2>
            <p><em>Action recommandée : Contacter ces patients pour planifier un suivi rapproché</em></p>
            
            <table>
                <thead>
                    <tr>
                        <th>Rang</th>
                        <th>Patient ID</th>
                        <th>Score de Risque</th>
                        <th>Probabilité</th>
                        <th>Catégorie</th>
                        <th>Action recommandée</th>
                    </tr>
                </thead>
                <tbody>
    """
    
    # Lignes du tableau
    for idx, (_, patient) in enumerate(patients_sorted.iterrows(), 1):
        risk_score = patient['risk_score_100']
        
        # Recommandation basée sur le score
        if risk_score >= 85:
            action = "🔴 URGENT - Appel immédiat"
        elif risk_score >= 75:
            action = "🟠 Contact sous 24h"
        else:
            action = "🟡 Suivi dans 48h"
        
        html += f"""
                    <tr>
                        <td><strong>#{idx}</strong></td>
                        <td>{patient['patient_id']}</td>
                        <td class="risk-high">{risk_score:.1f}/100</td>
                        <td>{patient['risk_probability']:.1%}</td>
                        <td><span class="risk-high">{patient['risk_category']}</span></td>
                        <td>{action}</td>
                    </tr>
        """
    
    # Pied de page
    html += f"""
                </tbody>
            </table>
            
            <div class="footer">
                <p><strong>Note importante :</strong></p>
                <ul>
                    <li>Cette alerte est générée automatiquement par le modèle prédictif (Version 5)</li>
                    <li>Les scores sont calculés en temps réel à partir des données patients</li>
                    <li>Pour plus de détails, consultez le tableau de bord Power BI</li>
                    <li>En cas de question, contactez l'équipe Data Science</li>
                </ul>
                <p style="color: #666; margin-top: 20px;">
                    <em>Cet e-mail a été envoyé automatiquement - Ne pas répondre</em>
                </p>
            </div>
        </div>
    </body>
    </html>
    """
    
    return html

In [0]:
# CELLULE 16 : TEST D'ENVOI E-MAIL
print("=" * 70)
print("TEST D'ENVOI E-MAIL")
print("=" * 70)
# Créer des données de test simulées pour patients à haut risque
import pandas as pd
from datetime import datetime
# Créer 3 patients à haut risque fictifs pour le test
test_high_risk_patients = pd.DataFrame({
    'patient_id': ['P001', 'P002', 'P003'],
    'prediction': [1, 1, 1],
    'risk_probability': [0.92, 0.87, 0.78],
    'risk_score_100': [92.0, 87.0, 78.0],
    'risk_category': ['Élevé', 'Élevé', 'Élevé'],
    'readmission_reelle': [None, None, None],
    'prediction_timestamp': [datetime.now(), datetime.now(), datetime.now()],
    'model_name': ['hospital_readmissions_model_mlflow@champion'] * 3,
    'model_version': ['5', '5', '5']
})

print(f" Patients de test créés : {len(test_high_risk_patients)}")
print("\nAperçu des patients :")
print(test_high_risk_patients[['patient_id', 'risk_score_100', 'risk_category']])

print("\n" + "=" * 70)
print("ENVOI DE L'E-MAIL DE TEST...")
print("=" * 70)

try:
  
    send_high_risk_alert(test_high_risk_patients, config=EMAIL_CONFIG) 
except Exception as e:
    print("\n" + "=" * 70)
    print(" ERREUR LORS DE L'ENVOI")
    print("=" * 70)
    print(f"Erreur : {str(e)}")
    
    # Diagnostics détaillés
    if "Authentication" in str(e):
        print("\n PROBLÈME D'AUTHENTIFICATION")
        print("   Vérifiez que :")
        print("   1. Le mot de passe d'application est correct (sans espaces)")
        print("   2. L'authentification à 2 facteurs est activée")
        print(f"   3. Email utilisé : {EMAIL_CONFIG['sender_email']}")
    elif "Connection" in str(e):
        print("\n PROBLÈME DE CONNEXION")
        print("   Vérifiez votre connexion Internet")
    else:
        print("\n ERREUR INATTENDUE")
        import traceback
        traceback.print_exc()

TEST D'ENVOI E-MAIL
 Patients de test créés : 3

Aperçu des patients :
  patient_id  risk_score_100 risk_category
0       P001            92.0         Élevé
1       P002            87.0         Élevé
2       P003            78.0         Élevé

ENVOI DE L'E-MAIL DE TEST...
 E-mail d'alerte envoyé avec succès à 2 destinataire(s)
  Patients concernés : 3


In [0]:
# CELLULE 17 : CONFIGURATION JOB DATABRICKS 
print("="*70)
print("CONFIGURATION JOB DATABRICKS")
print("="*70)

print("""
JOB QUOTIDIEN RECOMMANDE

Nom du job : readmissions_daily_pipeline

Notebook :
/Users/<email>/09_PowerBI_Datamart

Commande :
predictions = batch_score_new_patients(n_patients=100)

Planification :
- Daily
- 07:00
- Timezone : Africa/Dakar

Cluster :
- Runtime : 13.3 LTS ML
- Workers : 2 à 4
- Mode : Standard
""")

print("="*70)


CONFIGURATION JOB DATABRICKS

JOB QUOTIDIEN RECOMMANDE

Nom du job : readmissions_daily_pipeline

Notebook :
/Users/<email>/09_PowerBI_Datamart

Commande :
predictions = batch_score_new_patients(n_patients=100)

Planification :
- Daily
- 07:00
- Timezone : Africa/Dakar

Cluster :
- Runtime : 13.3 LTS ML
- Workers : 2 à 4
- Mode : Standard



In [0]:
# CELLULE 18 : RÉCAPITULATIF FINAL
print("="*70)
print("RECAPITULATIF FINAL - NOTEBOOK 09")
print("="*70)

views_created = [
    "vw_readmissions_kpis_daily",
    "vw_readmissions_high_risk_patients",
    "vw_readmissions_trends",
    "vw_readmissions_model_comparison",
    "vw_readmissions_by_risk_category"
]

for i, v in enumerate(views_created, 1):
    print(f"{i}. {v}")

total_predictions = spark.table(PREDICTIONS_TABLE).count()

print(f"\nTotal predictions : {total_predictions:,}")
print("Pipeline Power BI operationnel")
print("Scoring automatise actif")
print("="*70)


RECAPITULATIF FINAL - NOTEBOOK 09
1. vw_readmissions_kpis_daily
2. vw_readmissions_high_risk_patients
3. vw_readmissions_trends
4. vw_readmissions_model_comparison
5. vw_readmissions_by_risk_category

Total predictions : 600
Pipeline Power BI operationnel
Scoring automatise actif
